In [80]:
import pandas as pd
import numpy as np

df = pd.read_csv("vietnam_housing_dataset.csv")
print(df.shape)
df.head(10)

(30229, 12)


,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,Have certificate,NaN,8.60
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.50
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,NaN,NaN,Sale contract,NaN,8.90
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,NaN,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,NaN,NaN,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90
5,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",91.0,7.0,13.0,Tây - Bắc,NaN,NaN,NaN,NaN,Have certificate,NaN,9.80
6,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",64.0,4.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.20
7,"Dự án Him Lam Thường Tín, Huyện Thường Tín, Hà...",74.0,5.0,18.0,Nam,Nam,5.0,4.0,5.0,Have certificate,NaN,9.90
8,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",48.0,NaN,NaN,NaN,NaN,5.0,6.0,NaN,NaN,Basic,5.70
9,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",91.0,7.0,NaN,NaN,NaN,5.0,NaN,NaN,Have certificate,NaN,9.50


In [81]:
print("=== THÔNG TIN CỘT ===")
print(df.info())

print("\n=== MISSING VALUES (%) ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(
    pd.DataFrame({"Missing": missing, "Missing %": missing_pct}).sort_values(
        "Missing %", ascending=False
    )
)

print("\n=== THỐNG KÊ MÔ TẢ ===")
df.describe()

=== THÔNG TIN CỘT ===
<class 'pandas.DataFrame'>
RangeIndex: 30229 entries, 0 to 30228
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Address            30229 non-null  str    
 1   Area               30229 non-null  float64
 2   Frontage           18665 non-null  float64
 3   Access Road        16932 non-null  float64
 4   House direction    8990 non-null   str    
 5   Balcony direction  5246 non-null   str    
 6   Floors             26626 non-null  float64
 7   Bedrooms           25067 non-null  float64
 8   Bathrooms          23155 non-null  float64
 9   Legal status       25723 non-null  str    
 10  Furniture state    16110 non-null  str    
 11  Price              30229 non-null  float64
dtypes: float64(7), str(5)
memory usage: 5.4 MB
None

=== MISSING VALUES (%) ===
                   Missing  Missing %
Balcony direction    24983      82.65
House direction      21239      70.26
Furniture sta

,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Price
count,30229.000000,18665.000000,16932.000000,26626.000000,25067.000000,23155.000000,30229.000000
mean,68.498741,5.361692,7.853800,3.410426,3.511030,3.346837,5.872078
std,48.069835,4.346174,7.451313,1.328897,1.309116,1.400181,2.211877
min,3.100000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,40.000000,4.000000,4.000000,2.000000,3.000000,2.000000,4.200000
50%,56.000000,4.500000,6.000000,3.000000,3.000000,3.000000,5.900000
75%,80.000000,5.000000,10.000000,4.000000,4.000000,4.000000,7.500000
max,595.000000,77.000000,85.000000,10.000000,9.000000,9.000000,11.500000


In [82]:
import re


def parse_address(addr):
    if pd.isna(addr):
        return pd.Series(
            {"Project": None, "Ward_Street": None, "District": None, "Province": None}
        )

    parts = [p.strip() for p in addr.split(",")]

    result = {"Project": None, "Ward_Street": None, "District": None, "Province": None}

    # 2 vị trí cuối: tỉnh, huyện
    if len(parts) >= 1:
        result["Province"] = parts[-1]
    if len(parts) >= 2:
        result["District"] = parts[-2]

    # Vị trí đầu: dự án hoặc không
    if len(parts) >= 1:
        first = parts[0]
        if first.lower().startswith("dự án"):
            result["Project"] = first
            middle = parts[1:-2]  # phần giữa sau dự án
        else:
            middle = parts[0:-2]  # phần giữa tính từ đầu

        # Gom phần giữa vào Ward_Street
        if middle:
            result["Ward_Street"] = ", ".join(middle)

    return pd.Series(result)


addr_parsed = df["Address"].apply(parse_address)
df = pd.concat([df, addr_parsed], axis=1)
df.head(5)

,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price,Project,Ward_Street,District,Province
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,Have certificate,NaN,8.60,Dự án The Empire - Vinhomes Ocean Park 2,Xã Long Hưng,Văn Giang,Hưng Yên
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.50,Dự án The Crown - Vinhomes Ocean Park 3,Xã Nghĩa Trụ,Văn Giang,Hưng Yên
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,NaN,NaN,Sale contract,NaN,8.90,Dự án The Crown - Vinhomes Ocean Park 3,Xã Nghĩa Trụ,Văn Giang,Hưng Yên
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,NaN,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35,NaN,"Đường Nguyễn Văn Khối, Phường 11",Gò Vấp,Hồ Chí Minh
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,NaN,NaN,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90,NaN,"Đường Quang Trung, Phường 8",Gò Vấp,Hồ Chí Minh


In [83]:
numeric_cols = [
    "Area",
    "Frontage",
    "Access Road",
    "Floors",
    "Bedrooms",
    "Bathrooms",
    "Price",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)
print(df[numeric_cols].describe())

Area           float64
Frontage       float64
Access Road    float64
Floors         float64
Bedrooms       float64
Bathrooms      float64
Price          float64
dtype: object
               Area      Frontage   Access Road        Floors      Bedrooms  \
count  30229.000000  18665.000000  16932.000000  26626.000000  25067.000000   
mean      68.498741      5.361692      7.853800      3.410426      3.511030   
std       48.069835      4.346174      7.451313      1.328897      1.309116   
min        3.100000      1.000000      1.000000      1.000000      1.000000   
25%       40.000000      4.000000      4.000000      2.000000      3.000000   
50%       56.000000      4.500000      6.000000      3.000000      3.000000   
75%       80.000000      5.000000     10.000000      4.000000      4.000000   
max      595.000000     77.000000     85.000000     10.000000      9.000000   

          Bathrooms         Price  
count  23155.000000  30229.000000  
mean       3.346837      5.872078  
std  

In [84]:
# Chuẩn hóa text: strip whitespace, title case
cat_cols = ["House direction", "Balcony direction", "Legal status", "Furniture state"]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace("nan", np.nan)

# Kiểm tra các giá trị unique
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- House direction ---
House direction
NaN           21239
Đông - Nam     1916
Đông - Bắc     1180
Tây - Nam      1152
Tây - Bắc      1125
Nam            1042
Đông            952
Bắc             832
Tây             791
Name: count, dtype: int64

--- Balcony direction ---
Balcony direction
NaN           24983
Đông - Nam     1087
Nam             701
Đông - Bắc      670
Tây - Nam       646
Tây - Bắc       640
Đông            577
Bắc             474
Tây             451
Name: count, dtype: int64

--- Legal status ---
Legal status
Have certificate    24774
NaN                  4506
Sale contract         949
Name: count, dtype: int64

--- Furniture state ---
Furniture state
NaN      14119
Full     10591
Basic     5519
Name: count, dtype: int64


In [85]:
def flag_outliers_iqr(series, factor=3.0):
    """Dùng IQR với factor rộng hơn (3.0) để chỉ flag outlier cực đoan"""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series < lower) | (series > upper)


outlier_cols = ["Area", "Frontage", "Access Road", "Price"]

for col in outlier_cols:
    mask = flag_outliers_iqr(df[col].dropna())
    n_outliers = mask.sum()
    print(f"{col}: {n_outliers} outliers ({n_outliers/len(df)*100:.2f}%)")
    if n_outliers > 0:
        print(df.loc[df[col].notna()][mask][[col, "Address"]].head(5))
        print()

Area: 619 outliers (2.05%)
      Area                                            Address
16   300.0  Dự án Khu đô thị Phương Đông, Xã Đông Xá, Vân ...
24   369.0  Dự án Vườn Vua Resort & Villas, Đường 316, Xã ...
41   378.0                  Xã Lại Thượng, Thạch Thất, Hà Nội
78   368.0  Dự án Vườn Vua Resort & Villas, Đường Trung Th...
208  272.0                  Xã Bình Minh, Trảng Bom, Đồng Nai

Frontage: 1134 outliers (3.75%)
    Frontage                                            Address
16      15.0  Dự án Khu đô thị Phương Đông, Xã Đông Xá, Vân ...
24      15.0  Dự án Vườn Vua Resort & Villas, Đường 316, Xã ...
37      17.0  Dự án Hoàng Nam Uyên Hưng, Đường Tố Hữu, Phườn...
48      10.0  Đường Quốc Lộ 17B, Phường Minh Tân, Kinh Môn, ...
73      10.0  Dự án NovaWorld Phan Thiết, Đường Lạc Long Quâ...

Access Road: 468 outliers (1.55%)
    Access Road                                            Address
16         32.0  Dự án Khu đô thị Phương Đông, Xã Đông Xá, Vân ...
37         36.0

In [86]:
# --- Chiến lược impute ---
# Số: dùng MEDIAN theo Province (tránh bị kéo bởi outlier)
# Categorical: giữ NaN (không đoán mò hướng nhà / pháp lý)

for col in ["Area", "Frontage", "Access Road", "Floors", "Bedrooms", "Bathrooms"]:
    df[col] = df.groupby("Province")[col].transform(lambda x: x.fillna(x.median()))
    # Nếu vẫn còn NaN (tỉnh chỉ có 1 dòng), fallback về median toàn bộ
    df[col] = df[col].fillna(df[col].median())

print("Missing sau impute:")
print(df[numeric_cols].isnull().sum())

Missing sau impute:
Area           0
Frontage       0
Access Road    0
Floors         0
Bedrooms       0
Bathrooms      0
Price          0
dtype: int64


In [87]:
# Giá trên m2
df["Price_per_m2"] = (df["Price"] / df["Area"]).round(4)

# Loại bất động sản: dự án hay nhà phố
df["Is_Project"] = df["Project"].notna().astype(int)

# Nhóm diện tích
df["Area_group"] = pd.cut(
    df["Area"],
    bins=[0, 50, 80, 120, 200, float("inf")],
    labels=["<50m²", "50-80m²", "80-120m²", "120-200m²", ">200m²"],
)

# Có đầy đủ pháp lý không
df["Has_certificate"] = (df["Legal status"] == "Have certificate").astype(int)

print(
    df[
        ["Price", "Area", "Price_per_m2", "Is_Project", "Area_group", "Has_certificate"]
    ].head(10)
)

   Price  Area  Price_per_m2  Is_Project Area_group  Has_certificate
0   8.60  84.0        0.1024           1   80-120m²                1
1   7.50  60.0        0.1250           1    50-80m²                0
2   8.90  90.0        0.0989           1   80-120m²                0
3   5.35  54.0        0.0991           0    50-80m²                1
4   6.90  92.0        0.0750           0   80-120m²                1
5   9.80  91.0        0.1077           1   80-120m²                1
6   7.20  64.0        0.1125           1    50-80m²                0
7   9.90  74.0        0.1338           1    50-80m²                1
8   5.70  48.0        0.1188           1      <50m²                0
9   9.50  91.0        0.1044           1   80-120m²                1


In [88]:
n_before = len(df)
df = df.drop_duplicates()
n_after = len(df)
print(f"Removed {n_before - n_after} duplicate rows. Remaining: {n_after}")

Removed 43 duplicate rows. Remaining: 30186


In [89]:
print("=== FINAL SHAPE ===")
print(df.shape)

print("\n=== MISSING VALUES SAU CLEANING ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== SAMPLE ===")
df.head(5)

=== FINAL SHAPE ===
(30186, 20)

=== MISSING VALUES SAU CLEANING ===
House direction      21203
Balcony direction    24943
Legal status          4495
Furniture state      14095
Project              26549
Ward_Street            115
District                 3
dtype: int64

=== SAMPLE ===


,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price,Project,Ward_Street,District,Province,Price_per_m2,Is_Project,Area_group,Has_certificate
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,5.0,13.0,NaN,NaN,4.0,6.0,4.0,Have certificate,NaN,8.60,Dự án The Empire - Vinhomes Ocean Park 2,Xã Long Hưng,Văn Giang,Hưng Yên,0.1024,1,80-120m²,1
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,5.0,13.0,NaN,NaN,5.0,6.0,4.0,NaN,NaN,7.50,Dự án The Crown - Vinhomes Ocean Park 3,Xã Nghĩa Trụ,Văn Giang,Hưng Yên,0.1250,1,50-80m²,0
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,6.0,4.0,Sale contract,NaN,8.90,Dự án The Crown - Vinhomes Ocean Park 3,Xã Nghĩa Trụ,Văn Giang,Hưng Yên,0.0989,1,80-120m²,0
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,4.1,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35,NaN,"Đường Nguyễn Văn Khối, Phường 11",Gò Vấp,Hồ Chí Minh,0.0991,0,50-80m²,1
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,4.1,6.0,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90,NaN,"Đường Quang Trung, Phường 8",Gò Vấp,Hồ Chí Minh,0.0750,0,80-120m²,1


In [90]:
# # Impute Bedrooms và Bathrooms theo Area_group + Province
# # Vì nhà cùng diện tích, cùng khu vực thường có số phòng tương tự

# for col in ["Bedrooms", "Bathrooms"]:
#     df[col] = df.groupby(["Province", "Area_group"])[col].transform(
#         lambda x: x.fillna(x.median())
#     )
#     # Fallback: nếu nhóm quá nhỏ, dùng median theo Province
#     df[col] = df.groupby("Province")[col].transform(lambda x: x.fillna(x.median()))
#     # Fallback cuối: median toàn bộ
#     df[col] = df[col].fillna(df[col].median())

# # Sau đó làm tròn về số nguyên (không có 2.5 phòng ngủ)
# df["Bedrooms"] = df["Bedrooms"].round().astype(int)
# df["Bathrooms"] = df["Bathrooms"].round().astype(int)

In [91]:
# Lưu file đã clean
df.to_csv("vietnam_housing_dataset_cleaned.csv", index=False, encoding="utf-8-sig")